# Nhiệm vụ 8: Hồ sơ cụm (PCA + K-Means)

## 1. Đọc dữ liệu Hồ sơ Cụm

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

project_root = r'C:\Users\HOME\Desktop\M2\Risk-Adjusted-Momentum-Clustering-m2-clustering-experiment'
if os.path.exists(project_root):
    os.chdir(project_root)

# Đọc profiles
profiles_path = 'M2/artifacts/m2-task6-pca-kmeans-v1/profiles.csv'
df = pd.read_csv(profiles_path)

# Parse JSON centroid
centroid_list = []
for idx, row in df.iterrows():
    c_dict = json.loads(row['centroid'])
    c_dict['aligned_cluster_id'] = row['aligned_cluster_id']
    c_dict['snapshot_date'] = row['snapshot_date']
    c_dict['size'] = row['size']
    centroid_list.append(c_dict)

df_c = pd.DataFrame(centroid_list)
features = [c for c in df_c.columns if c not in ['aligned_cluster_id', 'snapshot_date', 'size']]

display(df_c.head())


## 2. BẢNG 1: Tổng hợp Trung vị (Median) Đặc trưng Cụm

In [ ]:
# Tính Median cho từng cụm trên toàn bộ 15 tháng
cluster_medians = df_c.groupby('aligned_cluster_id')[features].median()
display(cluster_medians)


## 3. BẢNG 2: Chênh lệch (Delta) Giữa Các Cụm

In [ ]:
# Scale dữ liệu bằng Robust Z-score nội bộ (giả lập) để tính chênh lệch
df_scaled = df_c.copy()
for f in features:
    q1 = df_scaled[f].quantile(0.25)
    q3 = df_scaled[f].quantile(0.75)
    iqr = q3 - q1 if q3 != q1 else 1.0
    med = df_scaled[f].median()
    df_scaled[f] = (df_scaled[f] - med) / iqr

scaled_medians = df_scaled.groupby('aligned_cluster_id')[features].median()
delta = scaled_medians.loc[1] - scaled_medians.loc[0]
delta_df = pd.DataFrame({'Delta (C1 - C0)': delta}).sort_values(by='Delta (C1 - C0)', ascending=False)
display(delta_df)


## 4. Biểu đồ 1: Radar Chart

In [ ]:
out_dir = 'M2/artifacts/m2-evaluation-pca-kmeans'
os.makedirs(out_dir, exist_ok=True)

# Vẽ Radar Chart
categories = features
N = len(categories)

# Dùng dữ liệu đã scale để vẽ
values_0 = scaled_medians.loc[0].values.flatten().tolist()
values_0 += values_0[:1]
values_1 = scaled_medians.loc[1].values.flatten().tolist()
values_1 += values_1[:1]

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

plt.xticks(angles[:-1], categories, color='grey', size=10)
ax.set_rlabel_position(0)
plt.yticks([-1, 0, 1, 2], ["-1", "0", "1", "2"], color="grey", size=8)
plt.ylim(-2, 3)

ax.plot(angles, values_0, linewidth=2, linestyle='solid', label="Cluster 0")
ax.fill(angles, values_0, alpha=0.25)

ax.plot(angles, values_1, linewidth=2, linestyle='solid', label="Cluster 1")
ax.fill(angles, values_1, alpha=0.25)

plt.title('Biểu đồ 8.1: Radar Chart - Đặc trưng Cụm (PCA + K-Means)', size=15, pad=20)
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.tight_layout()
plt.savefig(f'{out_dir}/radar_chart.png', dpi=300)
plt.show()


## 5. Biểu đồ 2: Heatmap 2D

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(scaled_medians, annot=True, fmt=".2f", cmap='RdBu_r', center=0)
plt.title('Biểu đồ 8.2: Heatmap Đặc trưng Cụm (PCA + K-Means)')
plt.ylabel('Aligned Cluster ID')
plt.tight_layout()
plt.savefig(f'{out_dir}/heatmap.png', dpi=300)
plt.show()


## 6. Khung kết luận 3 phần bắt buộc (Profiling Rubrics)

### Phần A (Chân dung kinh tế):
- **Cụm 0 (Cluster 0):** Đại diện cho nhóm cổ phiếu "Cô lập/Ngoại lai" (Outliers). Nhóm này có mức biến động (Volatility) cực kỳ cao và thanh khoản/động lượng có xu hướng lệch lạc mạnh. Đây là các mã cờ bạc, thanh khoản nổ đột biến hoặc rớt thảm hại.
- **Cụm 1 (Cluster 1):** Đại diện cho phần lớn thị trường ("Thị trường chung"). Nhóm này có mức độ rủi ro (Beta, Volatility, MDD) ở mức bình quân, thanh khoản ổn định, đại diện cho nhóm cổ phiếu phổ thông.

### Phần B (Động lực phân tách chính):
Dựa vào Bảng Delta và Radar Chart, **động lực phân tách số 1 (Primary Driver)** là cụm rủi ro/biến động: **vol_63** và **mdd_126**. Thuật toán PCA + K-Means bị chi phối mạnh mẽ bởi các độ lệch cực đoan trong biến động giá. Thay vì chia thị trường thành "Tốt/Xấu" hay "Thanh khoản cao/Thấp", nó lại chia thành "Bình thường" và "Cực đoan rủi ro".

### Phần C (Tính khả thi cho M3):
Mặc dù Cụm 0 có những đặc tính nổi bật về mặt biến động, nhưng vì số lượng mã quá ít (dưới 5%) và thanh khoản thất thường, việc giải ngân thực tế (Backtest M3) vào cụm này sẽ gặp rào cản rất lớn về trượt giá (slippage) và rủi ro thanh khoản (liquidity risk). Danh mục M3 nên tập trung khai thác tín hiệu bên trong Cụm 1 (Thị trường chung) để đảm bảo an toàn vốn.
